# Комбинаторика и подсчёты

### *Лекция для тех, кто когда-нибудь пытался угадать пароль*

> Привет! На прошлой лекции мы разобрались с множествами — «коробками с вещами». Сегодня узнаем, **сколько способов** разложить эти вещи по-разному.
>
> Если ты когда-нибудь:
> - выбирал 3 пиццы из 10 в меню — ты уже занимался комбинаторикой;
> - пытался вспомнить 4-значный пин-код — ты решал задачу перебора;
> - расставлял 5 друзей за столом на 5 стульев — ты считал перестановки.
>
> Осталось только перевести это на язык формул. Без паники: на каждый символ будет картинка и жизненный пример.
>
> В конце — **тест из 10 вопросов** простым языком, без формул в вариантах ответа.

**Уровень:** продолжаем с нуля. Достаточно знать, что такое множество (см. Лекцию 1).

**Время:** ~2 часа. Можно с чаем и пиццей. 🍕

---

## План лекции

| # | Тема | Жизненная аналогия |
|---|---|---|
| 1 | Что такое комбинаторика | Меню пиццы и замок на двери |
| 2 | Три типа комбинаций | Обзор: перестановки / сочетания / размещения |
| 3 | Факториал n! | Сколько способов рассадить гостей |
| 4 | Перестановки | 3 стакана шулера на столе |
| 5 | Размещения (без повторений) | Кто какое место займёт |
| 6 | Сочетания | Корзина с лимонадом |
| 7 | Размещения с повторениями | Кодовый замок в подъезде |
| 8 | Сочетания с повторениями | Покупка лимонада с повторами |
| 9 | Подсчёт против перечисления | Сколько vs какие |
| 10 | Задача 1 — Счастливые билеты | 55 252 билетика |
| 11 | Задача 2 — Счастливые пираты | Делим n монет |
| 12 | Задача 3 — Bruteforce | Подбираем пароль |
| 13 | Зачем это в ML | Гиперпараметры и сэмплинг |
| 14 | Итоговый тест | 10 вопросов

---

## Подготовка окружения

In [ ]:
# ============================================================
# Подготовка окружения
# ============================================================
# Это как разложить инструменты на столе перед готовкой.
# Каждая библиотека — отдельный инструмент.

import math                          # Встроенный калькулятор: factorial, comb, sqrt
import itertools                     # Генератор всех комбинаций (перестановки/сочетания)
import random                        # Кубик для случайных чисел
from fractions import Fraction       # Дроби без потери точности

import numpy as np                   # Массивы и векторы
import matplotlib.pyplot as plt      # Холст для рисования
from matplotlib.patches import Circle, FancyBboxPatch, FancyArrowPatch, Rectangle

# Настраиваем шрифт, чтобы русские буквы отображались
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False  # минус не должен быть дефисом
plt.rcParams['figure.dpi'] = 110            # чёткость картинок

# Фиксируем случайность, чтобы картинки не "прыгали" при перезапуске
random.seed(42)                     # кубик всегда падает одинаково
np.random.seed(42)

print("Профессор готов. Заваривай чай.")

---

## 1. Что такое комбинаторика?

### Жизненная аналогия: меню пиццы

Зашёл ты в пиццерию. В меню **10 разных пицц**. Хочешь заказать **3 разные**. Способов выбрать — куча. А сколько именно?

Или: у тебя **4-значный пароль** на телефоне. Сколько комбинаций нужно перебрать, чтобы точно его угадать? (Спойлер: 10 000 — это недолго для компьютера.)

**Комбинаторика** — это раздел математики, который отвечает на вопрос **«сколькими способами?»**.

### Типичные задачи

| Задача | Сколько вариантов? |
|---|---|
| Счастливые билеты из 6 цифр | 55 252 (узнаем в задаче 1!) |
| 4-значный пин-код | 10 000 |
| Пароль из 8 латинских букв | 26⁸ ≈ 208 миллиардов |
| Распределить 5 гостей по 5 стульям | 120 |
| Выбрать 3 пиццы из 10 | 120 |
| Сгенерировать колоду из 52 карт | 52! ≈ 8·10⁶⁷ |

> Обрати внимание: пароль из 8 букв перебирается компьютером за секунды, а 52! — это число, превышающее количество атомов в наблюдаемой Вселенной. **Комбинаторика объясняет, почему пароли работают (или не работают).**

### Зачем это в ML

- **Генерация данных:** сколько аугментаций можно сделать из картинки? Поворот × сдвиг × цвет × шум = комбинаторный взрыв.
- **Hyperparameter search:** у тебя 5 гиперпараметров, у каждого по 4 значения → 4⁵ = 1024 конфигурации. Сколько обучать?
- **Сэмплинг:** из 1 000 000 датасета берём случайные 1000 подмножеств — это сочетания.
- **Ансамбли:** как выбрать 3 модели из 10, чтобы они дополняли друг друга? — это биномиальный коэффициент.

> Если ты не понимаешь, **сколько вариантов у твоей модели**, ты не сможешь оценить, насколько она устойчива.

---

## 2. Три типа комбинаций (карта местности)

Прежде чем нырять в формулы — нарисуем карту. Все задачи комбинаторики сводятся к одному из трёх типов.

### Большая таблица

| Тип | Что делаем | Порядок важен? | Повторения? | Пример |
|---|---|---|---|---|
| **Перестановка** | берём ВСЕ элементы по очереди | да | нет | 3 стакана шулера на столе |
| **Размещение** | берём НЕКОТОРЫЕ элементы по очереди | да | да/нет | кодовый замок |
| **Сочетание** | берём НЕКОТОРЫЕ элементы в мешок | нет | обычно нет | корзина с лимонадом |

### Запоминание за 30 секунд

**Перестановка** — это **«расставить всех по местам»**.
- 5 друзей, 5 стульев. Кто куда сядет?
- Важно: **все** на месте, **порядок важен**, **повторений нет** (один человек не сядет на 2 стула).

**Размещение** — это **«выбрать несколько и расставить по местам»**.
- 10 претендентов, нанимаем 3 на разные должности (директор, бухгалтер, уборщик).
- Важно: **порядок важен** (кто директор, а кто уборщик — разные ситуации).

**Сочетание** — это **«ссыпать выбранные в мешок»**.
- 10 сортов лимонада, берём 3 разных в корзину.
- Важно: **порядок НЕ важен** (в корзине всё равно, в каком порядке лежат бутылки).

> **Главный вопрос, который задаёт себе комбинаторик:** «Если я поменяю местами две вещи — это будет **та же** ситуация или **другая**?»
>
> - Если **та же** → это **сочетание**.
> - Если **другая** → это **размещение** или **перестановка**.

In [ ]:
# ============================================================
# Картинка 1: три типа комбинаций на примере {A, B, C}
# ============================================================
# Рисуем три панельки: перестановки / размещения / сочетания

fig, axes = plt.subplots(1, 3, figsize=(15, 5))  # 1 строка, 3 колонки

# --- Панель 1: Перестановки из {A, B, C} — все 6 ---
ax = axes[0]
perms = ['ABC', 'ACB', 'BAC', 'BCA', 'CAB', 'CBA']  # все 6 перестановок
for i, p in enumerate(perms):
    row, col = divmod(i, 2)  # 3 строки × 2 колонки
    # Каждый символ — отдельный квадратик
    for j, ch in enumerate(p):
        rect = Rectangle((col*2.2 + j*0.6, 2.5 - row*0.8), 0.5, 0.5,
                          facecolor='#3498db', alpha=0.7, edgecolor='#2c3e50')
        ax.add_patch(rect)
        ax.text(col*2.2 + j*0.6 + 0.25, 2.5 - row*0.8 + 0.25, ch,
                ha='center', va='center', fontsize=12, fontweight='bold', color='white')
ax.set_title('Перестановки\n3! = 6 способов', fontsize=12, fontweight='bold')
ax.set_xlim(-0.3, 4.5); ax.set_ylim(0, 3.3)
ax.axis('off')

# --- Панель 2: Размещения 2 из {A, B, C} — все 6 (порядок важен) ---
ax = axes[1]
arrs = ['AB', 'AC', 'BA', 'BC', 'CA', 'CB']  # A(3,2) = 6
for i, p in enumerate(arrs):
    row, col = divmod(i, 2)
    for j, ch in enumerate(p):
        rect = Rectangle((col*2.2 + j*0.6, 2.5 - row*0.8), 0.5, 0.5,
                          facecolor='#e67e22', alpha=0.7, edgecolor='#2c3e50')
        ax.add_patch(rect)
        ax.text(col*2.2 + j*0.6 + 0.25, 2.5 - row*0.8 + 0.25, ch,
                ha='center', va='center', fontsize=12, fontweight='bold', color='white')
ax.set_title('Размещения 2 из 3\nA(3,2) = 6 (порядок важен)', fontsize=12, fontweight='bold')
ax.set_xlim(-0.3, 4.5); ax.set_ylim(0, 3.3)
ax.axis('off')

# --- Панель 3: Сочетания 2 из {A, B, C} — всего 3 (порядок НЕ важен) ---
ax = axes[2]
combs = ['AB', 'AC', 'BC']  # C(3,2) = 3 (AB = BA, считаем один раз)
for i, p in enumerate(combs):
    for j, ch in enumerate(p):
        rect = Rectangle((j*0.6 + 0.5, 2.4 - i*0.8), 0.5, 0.5,
                          facecolor='#27ae60', alpha=0.7, edgecolor='#2c3e50')
        ax.add_patch(rect)
        ax.text(j*0.6 + 0.5 + 0.25, 2.4 - i*0.8 + 0.25, ch,
                ha='center', va='center', fontsize=12, fontweight='bold', color='white')
ax.set_title('Сочетания 2 из 3\nC(3,2) = 3 (порядок НЕ важен)', fontsize=12, fontweight='bold')
ax.set_xlim(0, 2.5); ax.set_ylim(0, 3.3)
ax.axis('off')

plt.suptitle('Три типа комбинаций из множества {A, B, C}',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 3. Факториал n! — кирпич комбинаторики

### Жизненная аналогия: рассадка гостей

У тебя **5 друзей** и **5 стульев** за столом. Сколькими способами их можно рассадить?

Подумаем пошагово:
1. На **первый** стул сядет один из 5 друзей → 5 вариантов.
2. На **второй** — один из 4 оставшихся → 4 варианта.
3. На **третий** — один из 3 → 3 варианта.
4. На **четвёртый** — один из 2 → 2 варианта.
5. На **пятый** — последний оставшийся → 1 вариант.

Итого: 5 × 4 × 3 × 2 × 1 = **120 способов**.

Это произведение и называется **факториалом**.

### Формула

$$n! = n \cdot (n-1) \cdot (n-2) \cdot \ldots \cdot 2 \cdot 1$$

Читается: **«эн факториал»**. Это произведение всех натуральных чисел от 1 до n.

### Разбираем по символам

| Символ | Что значит |
|---|---|
| $n$ | переменная — целое неотрицательное число |
| $!$ | восклицательный знак = «факториал» |
| $\cdot$ | умножение (точка по центру, не звёздочка) |
| $\ldots$ | «и так далее до» |

**Примеры:**
- $1! = 1$
- $2! = 2 \cdot 1 = 2$
- $3! = 3 \cdot 2 \cdot 1 = 6$
- $4! = 4 \cdot 3 \cdot 2 \cdot 1 = 24$
- $5! = 5 \cdot 4 \cdot 3 \cdot 2 \cdot 1 = 120$
- $10! = 3\,628\,800$

### Особый случай: $0! = 1$

Это **не опечатка**. Ноль факториал равен **единице**. Почему?

**Причина 1 (практическая).** Чтобы формулы работали без исключений. Например, $C(n, n) = \frac{n!}{n! \cdot 0!} = 1$ — «выбрать все n из n» есть ровно один способ. Если бы $0! = 0$, формула ломалась.

**Причина 2 (логическая).** «Сколько способов упорядочить пустое множество?» — ровно один: ничего не делать. Это и есть $0! = 1$.

> **Зачем это в ML.** Факториал — основа всех формул комбинаторики. Без него не посчитать ни сочетания, ни размещения. А от них — никуда в статистике и теории вероятностей.

In [ ]:
# ============================================================
# Реализуем факториал вручную и сверяем с math.factorial
# ============================================================

def factorial(n):
    """Считаем n! = n * (n-1) * ... * 1 через цикл.
    Это та же рассадка гостей: для 5 человек будет 5*4*3*2*1.
    """
    if n < 0:                          # факториал отрицательного не определён
        raise ValueError("n должно быть >= 0")
    if n == 0:                         # особый случай: 0! = 1
        return 1
    result = 1                         # начинаем с 1 (нейтральный элемент умножения)
    for i in range(1, n + 1):          # i пробегает 1, 2, ..., n
        result *= i                    # result = result * i (домножаем на следующее число)
    return result

# Сравниваем нашу функцию с встроенной math.factorial
for n in [0, 1, 2, 3, 4, 5, 6, 10, 15, 20]:
    mine = factorial(n)                # наш результат
    builtin = math.factorial(n)        # встроенный результат
    match = "OK" if mine == builtin else "FAIL"
    print(f"{n:>3}! = {mine:>20}  (math: {builtin:>20})  [{match}]")

In [ ]:
# ============================================================
# Картинка 2: как быстро растёт факториал
# ============================================================
# Сравниваем n! с обычной линейной функцией 10*n и квадратом n^2

xs = list(range(0, 16))                  # n от 0 до 15
fact_vals = [math.factorial(n) for n in xs]  # n!
lin_vals = [10 * n for n in xs]          # 10*n — для сравнения
sq_vals = [n * n for n in xs]            # n^2 — для сравнения

fig, ax = plt.subplots(figsize=(10, 5.5))

# Логарифмическая шкала по Y, иначе факториал «съест» остальные графики
ax.set_yscale('log')                     # логарифмическая шкала

ax.plot(xs, fact_vals, 'o-', color='#e74c3c', linewidth=2.5,
        markersize=9, label='n!  (факториал)')
ax.plot(xs, sq_vals, 's--', color='#3498db', linewidth=1.8,
        markersize=7, label='n²  (квадрат)')
ax.plot(xs, lin_vals, '^:', color='#27ae60', linewidth=1.8,
        markersize=7, label='10·n  (линейная)')

# Подписываем несколько значений факториала прямо на графике
for n in [5, 10, 15]:
    ax.annotate(f'{n}! = {math.factorial(n):,}',
                xy=(n, math.factorial(n)),
                xytext=(n - 1.5, math.factorial(n) * 3),
                fontsize=9, color='#e74c3c',
                arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=1))

ax.set_xlabel('n', fontsize=12)
ax.set_ylabel('значение (логарифмическая шкала)', fontsize=12)
ax.set_title('Как быстро растёт факториал\n(логарифмическая шкала — иначе не уместить)',
             fontsize=13, fontweight='bold')
ax.legend(loc='upper left', fontsize=11)
ax.grid(True, alpha=0.3, which='both')   # сетка и основная, и побочная
plt.tight_layout()
plt.show()

print()
print("Профессор: видишь, как факториал 'взлетает'?")
print(f"  10!  = {math.factorial(10):,}            — это около 3.6 млн")
print(f"  20!  = {math.factorial(20):,}  — это 2.4 квинтиллиона")
print(f"  52!  ≈ 8 × 10^67                       — больше, чем атомов в галактике")
print("Именно поэтому перебор 52! вариантов — это физически невозможно.")

---

### Мини-тест: факториал

Закрой ноутбук и ответь устно (или запиши в ячейке ниже):

1. Чему равно $6!$?
2. Чему равно $0!$?
3. Что больше: $7!$ или $5000$?
4. Сколько способов рассадить 4 друзей на 4 стульях?
5. У тебя 52 карты. Сколько способов перетасовать колоду? (Хотя бы порядок величины.)

<details>
<summary><b>Показать ответы</b></summary>

1. $6! = 720$
2. $0! = 1$ (особый случай, чтобы формулы работали)
3. $7! = 5040 > 5000$
4. $4! = 24$
5. $52! \approx 8 \times 10^{67}$ — больше, чем атомов в наблюдаемой Вселенной.

</details>

---

## 4. Перестановки

### Жизненная аналогия: 3 стакана шулера

У шулера на столе **3 стакана**. Он их переставляет и получает разные комбинации. Сколько всего способов переставить 3 стакана?

Если стаканы обозначить `A`, `B`, `C`, то возможны:
- `ABC, ACB, BAC, BCA, CAB, CBA`

Итого **6 способов**. Это и есть **3! = 6**.

### Определение

> **Перестановка** — это способ **упорядочить все n элементов** множества.

**Три обязательных условия:**
1. Используются **ВСЕ** элементы (никого не забываем).
2. **Порядок важен** (ABC ≠ BAC).
3. **Повторений нет** (один элемент не встречается дважды).

### Формула

$$P(n) = n!$$

Читается: **«число перестановок из n элементов равно n факториал»**.

### Примеры

| n | n! | Примеры множества |
|---|---|---|
| 1 | 1 | {A} — один элемент, одна перестановка |
| 2 | 2 | {A, B} → AB, BA |
| 3 | 6 | {A, B, C} → 6 способов |
| 4 | 24 | {A, B, C, D} → 24 способа |
| 5 | 120 | {A, B, C, D, E} → 120 способов |

### Когда это НЕ перестановка

- Если **порядок не важен** → это сочетание, а не перестановка.
- Если берёшь **не все** элементы → это размещение.
- Если элементы **могут повторяться** → это размещения с повторениями.

> **Зачем это в ML.** Алгоритм сортировки по сути строит перестановку. Сложность сортировки — это про то, как быстро перебрать (или почти перебрать) n! вариантов. 

In [ ]:
# ============================================================
# Перестановки: считаем количество и выписываем все
# ============================================================
# Работаем с множеством {1, 2, 3}

n = 3
elements = [1, 2, 3]

# 1. Считаем количество перестановок по формуле: n!
count = math.factorial(n)               # 3! = 6
print(f"Количество перестановок из {n} элементов: {n}! = {count}")

# 2. Выписываем все перестановки через itertools.permutations
all_perms = list(itertools.permutations(elements))
# itertools.permutations возвращает кортежи вида (1, 2, 3)

print(f"\nВсе {count} перестановок:")
for i, perm in enumerate(all_perms, start=1):
    # Преобразуем кортеж (1,2,3) в строку "123" для красоты
    print(f"  {i}. {perm}  ->  {''.join(map(str, perm))}")

# 3. Проверяем: действительно ли их 6?
assert len(all_perms) == math.factorial(n), "Что-то не сходится!"
print(f"\nПроверка: itertools дал {len(all_perms)} перестановок, формула даёт {count} — сходится!")

In [ ]:
# ============================================================
# Картинка 3: все 6 перестановок множества {1, 2, 3}
# ============================================================
# Рисуем 6 рядов по 3 шарика — каждую перестановку отдельно

elements = [1, 2, 3]
all_perms = list(itertools.permutations(elements))   # 6 кортежей

colors = {1: '#e74c3c', 2: '#3498db', 3: '#27ae60'}  # цвет для каждой цифры

fig, ax = plt.subplots(figsize=(9, 7))

# 6 рядов, по 3 шарика в каждом
for row_idx, perm in enumerate(all_perms):
    y = 5.5 - row_idx * 0.9                          # вертикальная позиция ряда
    for col_idx, val in enumerate(perm):
        x = col_idx * 1.2 + 1                        # горизонтальная позиция шарика
        circle = Circle((x, y), 0.4, color=colors[val], alpha=0.85, zorder=3)
        ax.add_patch(circle)
        ax.text(x, y, str(val), fontsize=14, ha='center', va='center',
                color='white', fontweight='bold', zorder=4)
    # Подпись справа
    perm_str = ''.join(map(str, perm))
    ax.text(5.2, y, f'→  {perm_str}', fontsize=12, va='center',
            color='#2c3e50', fontweight='bold')
    # Номер перестановки слева
    ax.text(0.2, y, f'#{row_idx + 1}', fontsize=11, va='center',
            color='#7f8c8d', fontweight='bold')

ax.set_xlim(-0.2, 7)
ax.set_ylim(0, 6.5)
ax.set_aspect('equal')
ax.axis('off')

ax.set_title('Все 6 перестановок множества {1, 2, 3}\nP(3) = 3! = 6',
             fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

---

## 5. Размещения (без повторений)

### Жизненная аналогия: президент, вице-президент и секретарь

В клубе **10 человек**. Нужно выбрать:
- **президента**
- **вице-президента**
- **секретаря**

Это **три разные должности** — порядок важен. Одно и то же лицо не может занимать две должности.

Считаем:
1. Президентом может быть любой из 10 → 10 вариантов.
2. Вице-президентом — любой из 9 оставшихся → 9 вариантов.
3. Секретарём — любой из 8 → 8 вариантов.

Итого: $10 \cdot 9 \cdot 8 = 720$ способов.

### Определение

> **Размещение** — это упорядоченная выборка из $m$ элементов множества из $n$ элементов.

**Три условия:**
1. **Порядок важен** (AB ≠ BA).
2. Берём **m из n** (не обязательно все).
3. **Без повторений** (один элемент — один раз).

### Формула

$$A(n, m) = \frac{n!}{(n - m)!}$$

Читается: **«число размещений из n по m равно n факториал, делённый на n минус m факториал»**.

### Разбираем формулу по шагам

Почему именно так?

1. Всего перестановок всех $n$ элементов — это $n!$.
2. Но нам нужны только первые $m$ позиций. Остальные $(n-m)$ — нам не важны.
3. Поэтому делим $n!$ на $(n-m)!$ — это «отменяет» перестановки ненужных последних $(n-m)$ элементов.

**Проверка на примере:**
- $A(3, 2) = \frac{3!}{(3-2)!} = \frac{6}{1} = 6$ → из {A,B,C} выбираем по 2: AB, AC, BA, BC, CA, CB → 6 способов ✓

**Граничные случаи:**
- $A(n, 0) = 1$ — выбрать 0 элементов можно одним способом (ничего не выбрать).
- $A(n, 1) = n$ — выбрать 1 элемент: $n$ вариантов.
- $A(n, n) = n!$ — выбрать все $n$ элементов в определённом порядке = перестановка.

### Bruteforce (грубая сила)

Методы, связанные с перебором комбинаций (например, подбор пароля), называют **bruteforce**. Если пароль — это размещение с повторениями (см. §7), то число вариантов = $n^m$.

> **Зачем это в ML.** Когда ты выбираешь топ-3 модели из 10, чтобы собрать ансамбль (1-я модель как основная, 2-я как дополнение, 3-я как корректировка) — это размещение $A(10, 3) = 720$. Если бы порядок был не важен — было бы $C(10, 3) = 120$.

In [ ]:
# ============================================================
# Размещения: считаем и выписываем
# ============================================================
# Множество {0, 1, 2, 3}, выбираем по 2 (порядок важен)

n, m = 4, 2
elements = [0, 1, 2, 3]

# 1. По формуле: A(n, m) = n! / (n - m)!
def arrangements(n, m):
    """Число размещений из n по m без повторений."""
    return math.factorial(n) // math.factorial(n - m)   # // — целочисленное деление

count = arrangements(n, m)
print(f"A({n}, {m}) = {n}! / ({n}-{m})! = {math.factorial(n)} / {math.factorial(n - m)} = {count}")

# 2. Выписываем все размещения через itertools.permutations(., m)
all_arrs = list(itertools.permutations(elements, m))
# ВАЖНО: permutations(elements, m) даёт упорядоченные выборки длины m

print(f"\nВсе {count} размещений из {{0,1,2,3}} по 2:")
for i, arr in enumerate(all_arrs, start=1):
    print(f"  {i:>2}. {arr}")

# 3. Сравниваем с math.perm (доступно с Python 3.8)
builtin = math.perm(n, m)
print(f"\nПроверка через math.perm({n}, {m}) = {builtin}  — {'совпадает' if builtin == count else 'РАСХОЖДЕНИЕ'}")

# 4. Сколько способов выбрать президента, вице- и секретаря из 10 человек?
print(f"\nПрезидент+вице+секретарь из 10 человек: A(10, 3) = {arrangements(10, 3)}")

In [ ]:
# ============================================================
# Картинка 4: размещения 2 из 4 (12 способов) — укороченный показ
# ============================================================
elements = ['A', 'B', 'C', 'D']
all_arrs = list(itertools.permutations(elements, 2))   # A(4,2) = 12

colors = {'A': '#e74c3c', 'B': '#3498db', 'C': '#27ae60', 'D': '#f39c12'}

fig, ax = plt.subplots(figsize=(11, 7))

# 12 рядов по 2 шарика (4 столбца × 3 строки для компактности)
for idx, arr in enumerate(all_arrs):
    row = idx // 4         # 0, 1, 2
    col = idx % 4          # 0, 1, 2, 3
    base_x = col * 3.0     # горизонтальный сдвиг колонки
    base_y = 2.5 - row * 1.1  # вертикальная позиция ряда

    # Рисуем 2 шарика
    for j, ch in enumerate(arr):
        x = base_x + j * 0.9
        circle = Circle((x, base_y), 0.35, color=colors[ch], alpha=0.85, zorder=3)
        ax.add_patch(circle)
        ax.text(x, base_y, ch, fontsize=13, ha='center', va='center',
                color='white', fontweight='bold', zorder=4)

    # Подпись справа
    arr_str = ''.join(arr)
    ax.text(base_x + 2.0, base_y, f'→ {arr_str}',
            fontsize=11, va='center', color='#2c3e50')

ax.set_xlim(-0.5, 14)
ax.set_ylim(-0.5, 3.5)
ax.set_aspect('equal')
ax.axis('off')

ax.set_title('Все 12 размещений 2 из {A, B, C, D}\nA(4, 2) = 4! / (4-2)! = 24 / 2 = 12',
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

---

## 6. Сочетания

### Жизненная аналогия: корзина с лимонадом

В магазине **5 видов лимонада**:
- 🍐 грушевый
- 🍊 апельсиновый
- 🍒 вишнёвый
- 🍏 яблочный
- 🍒 черешневый

Хочешь взять **3 разные бутылки** в корзину. Поскольку на кассе платишь одинаково, **порядок не важен** — главное, какие 3 выбрал.

Сколькими способами?

### Определение

> **Сочетание** — это подмножество из $m$ элементов множества из $n$ элементов. **Порядок не важен.**

**Условия:**
1. **Порядок НЕ важен** ({A, B} = {B, A}).
2. Берём $m$ из $n$.
3. **Без повторений** (по умолчанию).

### Выводим формулу шаг за шагом

Это самая красивая часть лекции. Следи за руками.

**Шаг 1.** Представь, что порядок **важен**. Тогда выбор 3 лимонадов из 5 — это размещение $A(5, 3) = \frac{5!}{(5-3)!} = \frac{120}{2} = 60$.

**Шаг 2.** Но в этой 60 мы посчитали каждую корзину **несколько раз**. Например, корзину {груша, вишня, яблоко} посчитали и как (груша, вишня, яблоко), и как (вишня, груша, яблоко), и как (яблоко, груша, вишня) — все $3! = 6$ перестановок одной и той же тройки.

**Шаг 3.** Чтобы получить число **разных** корзин, делим 60 на 6: $\frac{60}{6} = 10$.

**Итог:**

$$C(5, 3) = \frac{A(5, 3)}{3!} = \frac{5!}{(5-3)! \cdot 3!} = \frac{5!}{3! \cdot 2!} = \frac{120}{6 \cdot 2} = 10$$

### Общая формула (биномиальный коэффициент)

$$C(n, m) = \frac{n!}{m! \cdot (n - m)!}$$

Читается: **«цэ из эн по эм»** или **«биномиальный коэффициент»**. Также обозначается $\binom{n}{m}$.

### Разбираем по символам

| Часть | Что значит |
|---|---|
| $C(n, m)$ | «число сочетаний из n по m» |
| $n!$ | все возможные порядки всех n элементов |
| $m!$ | убираем «лишние» порядки внутри выбранной группы (их все считаем за 1) |
| $(n-m)!$ | убираем «лишние» порядки среди невыбранных |
| $\frac{\ldots}{\ldots}$ | делим, чтобы убрать дубли |

### Граничные случаи

- $C(n, 0) = 1$ — один способ ничего не выбрать.
- $C(n, 1) = n$ — $n$ способов выбрать один элемент.
- $C(n, n) = 1$ — один способ выбрать все.
- $C(n, m) = C(n, n-m)$ — знаменитая симметрия: выбрать $m$ включить = выбрать $n-m$ исключить.

### Пример: 10 видов пиццы, берём 3

$$C(10, 3) = \frac{10!}{3! \cdot 7!} = \frac{3628800}{6 \cdot 5040} = 120$$

> 120 способов выбрать 3 пиццы. Если ты думал, что вариантов «не так уж много» — вот тебе факт: тебя ждёт 120 разных вечеров.

In [ ]:
# ============================================================
# Сочетания: считаем и выписываем
# ============================================================
# Пример с лимонадом: 5 видов, берём 3

lemonades = ['груша', 'апельсин', 'вишня', 'яблоко', 'черешня']
n, m = 5, 3

# 1. По формуле: C(n, m) = n! / (m! * (n - m)!)
def combinations(n, m):
    """Число сочетаний из n по m без повторений (биномиальный коэффициент)."""
    return math.factorial(n) // (math.factorial(m) * math.factorial(n - m))

count = combinations(n, m)
print(f"C({n}, {m}) = {n}! / ({m}! · ({n}-{m})!) = {math.factorial(n)} / ({math.factorial(m)} · {math.factorial(n-m)}) = {count}")

# 2. Выписываем все сочетания через itertools.combinations
all_combs = list(itertools.combinations(lemonades, m))
# itertools.combinations даёт НЕупорядоченные выборки (каждая тройка один раз)

print(f"\nВсе {count} корзин из 3 лимонадов:")
for i, comb in enumerate(all_combs, start=1):
    print(f"  {i:>2}. {comb}")

# 3. Сравниваем с math.comb
builtin = math.comb(n, m)
print(f"\nПроверка через math.comb({n}, {m}) = {builtin}  — {'совпадает' if builtin == count else 'РАСХОЖДЕНИЕ'}")

# 4. Симметрия: C(5, 3) должно равняться C(5, 2)
print(f"\nСимметрия: C(5, 3) = {combinations(5, 3)}, C(5, 2) = {combinations(5, 2)} — {'равны' if combinations(5, 3) == combinations(5, 2) else 'не равны'}")

In [ ]:
# ============================================================
# Картинка 5: треугольник Паскаля (числа C(n, m))
# ============================================================
# Треугольник Паскаля — это таблица значений C(n, m)
# Каждое число = сумма двух чисел над ним

ROWS = 9   # рисуем 9 рядов

fig, ax = plt.subplots(figsize=(11, 7))

for n in range(ROWS):
    for m in range(n + 1):
        val = math.comb(n, m)                    # C(n, m)
        # Координаты: по горизонтали - m, по вертикали - n
        # Сдвигаем каждую строку влево, чтобы получился треугольник
        x = m - n / 2
        y = -n
        # Размер круга пропорционален log(val+1)
        size = 0.4 + 0.3 * math.log(val + 1)
        # Цвет: чем больше val, тем насыщеннее
        intensity = min(1.0, math.log(val + 1) / 10)
        circle = Circle((x, y), size,
                         facecolor=(0.2 + 0.5 * intensity, 0.4 + 0.3 * intensity, 0.8),
                         edgecolor='#2c3e50', linewidth=1.5, zorder=3)
        ax.add_patch(circle)
        # Подписываем значение внутри
        ax.text(x, y, str(val), ha='center', va='center',
                fontsize=9, fontweight='bold', color='white', zorder=4)

ax.set_xlim(-5, 5)
ax.set_ylim(-ROWS + 0.5, 1.5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Треугольник Паскаля — значения C(n, m)\nКаждое число = сумма двух над ним',
             fontsize=13, fontweight='bold', pad=15)

# Добавим подписи рядов слева
for n in range(ROWS):
    ax.text(-5.3, -n, f'n={n}', fontsize=9, va='center', color='#7f8c8d')

plt.tight_layout()
plt.show()

# Выводим несколько значений для проверки
print("Первые ряды треугольника Паскаля:")
for n in range(8):
    row = [str(math.comb(n, m)) for m in range(n + 1)]
    print(f"  n={n}: {'  '.join(row)}")

---

### Мини-тест: размещения vs сочетания

1. Из 10 моделей нужно выбрать 3 для ансамбля (порядок не важен). Сколько способов?
2. Из 10 моделей выбрать: 1-я основная, 2-я доп., 3-я корректировка (порядок важен). Сколько способов?
3. Сколькими способами можно раздать 3 разные книги 3 друзьям (каждому по одной)?
4. В лотерее 6 из 49. Сколько возможных комбинаций?
5. Что больше: $C(10, 3)$ или $A(10, 3)$? Во сколько раз?

<details>
<summary><b>Показать ответы</b></summary>

1. $C(10, 3) = 120$
2. $A(10, 3) = 720$
3. Это $A(3, 3) = 3! = 6$ — перестановка 3 книг по 3 друзьям.
4. $C(49, 6) = 13\,983\,816$ — поэтому лотерея так выгодна устроителям.
5. $A(10, 3) = 720$, $C(10, 3) = 120$. Размещений больше в 6 раз ($= 3!$). Логично: для каждой тройки есть 6 порядков.

</details>

---

## 7. Размещения с повторениями

### Жизненная аналогия: кодовый замок в подъезде

На двери подъезда **4-значный кодовый замок**. Каждая цифра — от 0 до 9. **Цифры могут повторяться** (например, `0000` или `1122`).

Сколько всего комбинаций?

Считаем:
1. На 1-й позиции — 10 вариантов (0..9).
2. На 2-й позиции — снова 10 (можем повторить).
3. На 3-й — 10.
4. На 4-й — 10.

Итого: $10 \cdot 10 \cdot 10 \cdot 10 = 10^4 = 10\,000$ комбинаций.

### Определение

> **Размещение с повторениями** — упорядоченная выборка длины $m$ из $n$ элементов, **где элементы могут повторяться**.

### Формула

$$\overline{A}(n, m) = n^m$$

Читается: **«число размещений с повторениями из n по m равно n в степени m»**.

### Разбираем

- $n$ — сколько различных «символов» доступно (10 цифр, 26 букв и т.д.).
- $m$ — длина выборки (длина пароля).
- $n^m$ — потому что на каждой из $m$ позиций независимо $n$ вариантов.

### Примеры

| Задача | $n$ | $m$ | Вариантов |
|---|---|---|---|
| 4-значный пин-код | 10 | 4 | 10 000 |
| 8-буквенный пароль (только буквы) | 26 | 8 | $\approx 2 \cdot 10^{11}$ |
| 6-значный пароль (буквы+цифры) | 36 | 6 | $\approx 2.18 \cdot 10^9$ |
| Пароль из 8 символов (буквы+цифры+спецсимволы) | 70 | 8 | $\approx 5.76 \cdot 10^{14}$ |
| ДНК-последовательность длины 10 (A, T, G, C) | 4 | 10 | $4^{10} = 1\,048\,576$ |

### Парольная математика

Если хакер проверяет 1 миллиард паролей в секунду:
- 4-значный пин-код: 10 000 / 10⁹ = **0.00001 секунды** (взламывается мгновенно).
- 6-значный пароль из букв+цифр: 2.18·10⁹ / 10⁹ = **2.2 секунды**.
- 8-значный пароль из 70 символов: 5.76·10¹⁴ / 10⁹ = **5.76 дней**.
- 12-значный пароль из 70 символов: 70¹² ≈ 1.4·10²² / 10⁹ = **444 года**.

> **Вывод:** используй длинные пароли. Каждый дополнительный символ умножает число вариантов в $n$ раз.

In [ ]:
# ============================================================
# Размещения с повторениями
# ============================================================
# Считаем число паролей разной длины

def arrangements_with_repetition(n, m):
    """Число размещений с повторениями: n^m.
    n — размер алфавита, m — длина пароля.
    """
    return n ** m

# Сравниваем разные "пароли"
cases = [
    ("4-значный пин-код",                      10, 4),
    ("6-значный пин-код",                      10, 6),
    ("Пароль 8 букв (только lowercase)",       26, 8),
    ("Пароль 8 символов (буквы+цифры)",        36, 8),
    ("Пароль 8 символов (+ спецсимволы)",      70, 8),
    ("Пароль 12 символов (буквы+цифры+спец.)", 70, 12),
    ("Пароль 16 символов (буквы+цифры+спец.)", 70, 16),
]

print(f"{'Тип пароля':<42} {'n':>4} {'m':>4} {'Вариантов':>20} {'Время взлома':>20}")
print("-" * 100)
for name, n, m in cases:
    count = arrangements_with_repetition(n, m)   # сколько всего вариантов
    # Предположим, хакер проверяет 1 миллиард (1e9) паролей в секунду
    seconds = count / 1e9
    if seconds < 0.001:
        time_str = f"{seconds * 1000:.4f} мс"
    elif seconds < 1:
        time_str = f"{seconds * 1000:.1f} мс"
    elif seconds < 60:
        time_str = f"{seconds:.2f} с"
    elif seconds < 3600:
        time_str = f"{seconds / 60:.2f} мин"
    elif seconds < 86400:
        time_str = f"{seconds / 3600:.2f} ч"
    elif seconds < 86400 * 365:
        time_str = f"{seconds / 86400:.2f} дн"
    else:
        time_str = f"{seconds / (86400 * 365):.2e} лет"
    print(f"{name:<42} {n:>4} {m:>4} {count:>20,} {time_str:>20}")

In [ ]:
# ============================================================
# Картинка 6: как растёт число паролей с длиной
# ============================================================
# Для 4 алфавитов: цифры (10), буквы (26), буквы+цифры (36), все символы (70)

alphabets = [('цифры (n=10)', 10, '#e74c3c'),
             ('буквы (n=26)', 26, '#3498db'),
             ('буквы+цифры (n=36)', 36, '#27ae60'),
             ('все символы (n=70)', 70, '#9b59b6')]

lengths = list(range(1, 13))  # длина пароля от 1 до 12

fig, ax = plt.subplots(figsize=(11, 6))
ax.set_yscale('log')   # логарифмическая шкала — иначе не уместить

for name, n, color in alphabets:
    counts = [n ** m for m in lengths]            # число паролей для каждой длины
    ax.plot(lengths, counts, 'o-', color=color, linewidth=2.2,
            markersize=7, label=f'{name}')

# Подсветим "опасную зону" — пароли длиной 1-6
ax.axvspan(1, 6, alpha=0.08, color='red', label='опасная зона (взлом < 1 сек)')

ax.set_xlabel('длина пароля m', fontsize=12)
ax.set_ylabel('число комбинаций (логарифм)', fontsize=12)
ax.set_title('Сколько комбинаций у пароля разной длины\nКаждый +1 символ умножает число вариантов в n раз',
             fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3, which='both')
ax.set_xticks(lengths)
plt.tight_layout()
plt.show()

---

## 8. Сочетания с повторениями

### Жизненная аналогия: покупка лимонада (можно повторять)

Снова 5 видов лимонада. Но теперь ты берёшь **3 бутылки, и можно повторять** (например, 2 грушевых + 1 вишнёвый).

Сколько способов?

### Определение

> **Сочетание с повторениями** — выбор $m$ элементов из $n$ типов, где **один тип можно брать несколько раз**. Порядок не важен.

### Формула (звёзды и перегородки)

$$\overline{C}(n, m) = C(n + m - 1, m) = \frac{(n + m - 1)!}{m! \cdot (n - 1)!}$$

### Идея доказательства: метод «звёзд и перегородок»

Представь, что 3 выбранные бутылки — это 3 звёздочки: `★★★`. А между 5 видами лимонада — 4 перегородки `|`. Например:

| Запись | Что значит |
|---|---|
| `★★★\|\|\|\|` | 3 груши |
| `★\|★\|★\|\|` | 1 груша, 1 апельсин, 1 вишня |
| `★★\|\|★\|\|` | 2 груши, 1 вишня |
| `★\|\|\|★★\|` | 1 груша, 2 яблока |

Каждое распределение — это перестановка 3 звёзд и 4 перегородок (всего 7 позиций). Выбрать, **где стоят звёзды**, — это $C(7, 3) = 35$ способов.

**Итог:** $\overline{C}(5, 3) = C(5 + 3 - 1, 3) = C(7, 3) = 35$.

### Примеры

| Задача | $n$ | $m$ | Ответ |
|---|---|---|---|
| 3 лимонада из 5 (с повторениями) | 5 | 3 | 35 |
| 6 пончиков из 3 видов | 3 | 6 | 28 |
| 10 монет по 3 карманам | 3 | 10 | 66 |

> **Зачем это в ML.** Если ты распределяешь $m$ «очков важности» между $n$ классами — это сочетания с повторениями. Сколько способов расставить веса так, чтобы их сумма была фиксированной?

In [ ]:
# ============================================================
# Сочетания с повторениями
# ============================================================

def combinations_with_repetition(n, m):
    """Число сочетаний с повторениями из n по m.
    Формула: C(n + m - 1, m) = (n+m-1)! / (m! * (n-1)!)
    """
    return math.comb(n + m - 1, m)

# Пример: 3 лимонада из 5 видов, можно повторять
n, m = 5, 3
count = combinations_with_repetition(n, m)
print(f"C_with_rep({n}, {m}) = C({n}+{m}-1, {m}) = C({n+m-1}, {m}) = {count}")

# Выписываем все сочетания с повторениями через itertools.combinations_with_replacement
lemonades = ['груша', 'апельсин', 'вишня', 'яблоко', 'черешня']
all_combs_rep = list(itertools.combinations_with_replacement(lemonades, m))

print(f"\nВсе {count} корзин (с повторениями):")
for i, comb in enumerate(all_combs_rep, start=1):
    print(f"  {i:>2}. {comb}")

# Сравниваем: без повторений vs с повторениями
print(f"\n--- Сравнение ---")
print(f"Без повторений:  C({n}, {m})        = {math.comb(n, m)}")
print(f"С повторениями:  C_with_rep({n}, {m}) = {combinations_with_repetition(n, m)}")
print(f"Разница: {combinations_with_repetition(n, m) - math.comb(n, m)} дополнительных комбинаций")

---

## 9. Подсчёт против перечисления

### Главное отличие

| | Подсчёт | Перечисление |
|---|---|---|
| **Вопрос** | СКОЛЬКО? | КАКИЕ ИМЕННО? |
| **Пример** | Сколько счастливых билетов? | Выпиши все счастливые билеты |
| **Сложность** | Обычно $O(1)$ или $O(n)$ | Обычно $O(\text{результат})$ — пропорционально количеству |
| **В ML** | Сколько конфигураций у модели? | Сгенерируй все конфигурации |

### Четыре варианта

| | Подсчёт лёгкий | Подсчёт сложный |
|---|---|---|
| **Перечисление лёгкое** | 5 друзей по 5 стульям: считаем $5!=120$, выписываем 120 | Встречается редко |
| **Перечисление сложное** | Счастливые билеты: посчитать легко (формула), выписать 55 252 штук — долго | Колода карт: $52! \approx 10^{67}$ — ни посчитать нормально, ни выписать |

### Пример: подсчёт лёгкий, перечисление невозможно

Число перестановок колоды из 52 карт: $52! \approx 8.07 \times 10^{67}$.

- **Подсчёт:** одна строчка кода — `math.factorial(52)`.
- **Перечисление:** даже если бы компьютер выдавал по миллиарду перестановок в секунду, потребовалось бы $8 \times 10^{58}$ секунд ≈ $2.5 \times 10^{51}$ лет. Это в $10^{41}$ раз больше возраста Вселенной.

### Пример: перечисление лёгкое, подсчёт сложный

«Сколько способов разбить 100 на сумму натуральных чисел?» — это **функция разбиения** $p(n)$. Для $n = 100$ ответ 190 569 292 (вычисляется нетривиально), а выписать все разбиения — долгая, но понятная задача перебора.

### Практический вывод

- Если нужно **оценить время перебора** — достаточно подсчёта.
- Если нужно **получить конкретные варианты** — нужно перечисление.
- Если результат огромен — **перечисление невозможно**, нужно искать другие методы (эвристики, сэмплинг).

> **Зачем это в ML.** Подсчёт числа гиперпараметрических конфигураций скажет тебе, **сколько времени** займёт grid search. Если их $10^{12}$ — grid search не вариант, нужен random search или Bayesian optimization.

---

## 10. Задача 1 — Счастливые билеты

### Постановка

Билет из 6 цифр (от `000000` до `999999`) называется **счастливым**, если сумма первых трёх цифр равна сумме последних трёх.

**Примеры:**
- `123123` — счастливый: 1+2+3 = 1+2+3 = 6.
- `001001` — счастливый: 0+0+1 = 0+0+1 = 1.
- `123456` — несчастливый: 1+2+3=6, 4+5+6=15, не равны.

**Вопрос:** сколько всего счастливых билетов? (Спойлер: 55 252)

### Идея решения

Подсчёт:
1. Пробегаем все числа от `000000` до `999999` (всего миллион).
2. Для каждого числа разбиваем на 2 половины по 3 цифры.
3. Сравниваем суммы.
4. Считаем те, где суммы равны.

**Сложность:** $O(10^6)$ — это всего миллион итераций, компьютер сделает за доли секунды.

### Умная оптимизация (для тех, кто не хочет ждать)

Заметим: сумма трёх цифр принимает значения от 0 (000) до 27 (999). Посчитаем, **сколько троек цифр** дают каждую сумму:

| Сумма $s$ | Сколько троек дают $s$ |
|---|---|
| 0 | 1 (только 000) |
| 1 | 3 (001, 010, 100) |
| 2 | 6 (002, 011, 020, 101, 110, 200) |
| ... | ... |
| 27 | 1 (только 999) |

Тогда число счастливых билетов = $\sum_{s=0}^{27} (\text{число троек с суммой } s)^2$.

Это **квадратичная** свёртка — работает за $O(28)$ вместо $O(10^6)$.

In [ ]:
# ============================================================
# Задача 1: Счастливые билеты — наивный перебор
# ============================================================
# Перебираем все 1 000 000 билетов и считаем счастливые

def is_lucky(ticket_num):
    """Проверяем, счастливый ли билет (6 цифр).
    ticket_num — число от 0 до 999999.
    """
    # Превращаем число в 6-значную строку с ведущими нулями
    s = f"{ticket_num:06d}"                  # например, 123 -> "000123"
    # Первые 3 цифры и последние 3
    first3 = s[:3]                            # строка "000"
    last3 = s[3:]                             # строка "123"
    # Сумма цифр каждой половины
    sum_first = sum(int(c) for c in first3)   # 0+0+0 = 0
    sum_last = sum(int(c) for c in last3)     # 1+2+3 = 6
    return sum_first == sum_last

# Перебираем все билеты
lucky_count = 0
first_few_lucky = []                          # сохраним несколько для показа

for ticket in range(1_000_000):                # 0 .. 999999
    if is_lucky(ticket):
        lucky_count += 1
        if len(first_few_lucky) < 10:         # сохраняем первые 10
            first_few_lucky.append(f"{ticket:06d}")

print(f"Всего счастливых билетов: {lucky_count}")
print(f"Ожидаемый ответ: 55252  — {'совпадает' if lucky_count == 55252 else 'НЕ совпадает'}")
print(f"\nПервые 10 счастливых билетов: {first_few_lucky}")

In [ ]:
# ============================================================
# Задача 1 (продолжение): умный способ через распределение сумм
# ============================================================
# Считаем p[s] = сколько троек цифр дают сумму s, s от 0 до 27
# Тогда счастливых = sum(p[s]^2 for s in range(28))

from collections import Counter

# Считаем, сколько троек (a, b, c) дают каждую сумму
sum_counts = Counter()                         # словарь "сумма -> сколько троек"
for a in range(10):                            # первая цифра 0..9
    for b in range(10):                        # вторая
        for c in range(10):                    # третья
            sum_counts[a + b + c] += 1         # увеличиваем счётчик для этой суммы

# Печатаем распределение
print("Распределение сумм троек цифр:")
print(f"{'s':>3} | {'количество троек':>18}")
print("-" * 30)
for s in range(28):
    print(f"{s:>3} | {sum_counts[s]:>18}")

# Счастливых билетов = sum(p[s]^2)
lucky_smart = sum(cnt ** 2 for cnt in sum_counts.values())
print(f"\nУмный подсчёт: {lucky_smart}")
print(f"Наивный перебор: 55252")
print(f"Совпадает: {lucky_smart == 55252}")
print(f"\nУскорение: было 1 000 000 итераций, стало {10*10*10 + 28} = 1028 операций")

In [ ]:
# ============================================================
# Картинка 7: распределение сумм троек цифр и квадраты
# ============================================================
sums = list(range(28))
counts = [sum_counts[s] for s in sums]
squares = [c ** 2 for c in counts]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Левая панель: сколько троек дают сумму s
ax = axes[0]
ax.bar(sums, counts, color='#3498db', alpha=0.8, edgecolor='#2c3e50')
ax.set_xlabel('сумма трёх цифр s', fontsize=11)
ax.set_ylabel('сколько троек дают эту сумму', fontsize=11)
ax.set_title('Сколько троек (a,b,c) дают сумму s\nсимметрично вокруг s=13.5',
             fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Правая панель: квадраты — вклад в счастливые билеты
ax = axes[1]
ax.bar(sums, squares, color='#e74c3c', alpha=0.8, edgecolor='#2c3e50')
# Подсветим максимум
max_idx = squares.index(max(squares))
ax.bar(sums[max_idx], squares[max_idx], color='#f39c12', alpha=0.95,
       edgecolor='#2c3e50', linewidth=2)
ax.text(sums[max_idx], squares[max_idx] + 50,
        f'максимум: s={sums[max_idx]}\n{ squares[max_idx]:,} билетов',
        ha='center', fontsize=10, color='#2c3e50')
ax.set_xlabel('сумма s', fontsize=11)
ax.set_ylabel('вклад в число счастливых билетов (p[s]²)', fontsize=11)
ax.set_title(f'Вклад каждой суммы в счастливые билеты\nСумма всех столбиков = {sum(squares):,}',
             fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"\nПрофессор: видишь? Билеты с суммой 13 и 14 — самые 'популярные'.")
print(f"Именно поэтому 'счастливые' билеты встречаются не так уж редко.")

---

## 11. Задача 2 — Счастливые пираты

### Постановка

3 пирата делят между собой $n$ награбленных монет:
- 1-й хочет **не менее $a$%** от $n$.
- 2-й согласен на **не менее $b$%**.
- 3-й — на **не менее $c$%**.
- Сумма $a + b + c$ не превышает 100 (минимальные требования).

Если доли не дотягивают до минимальных — пираты ссорятся. Найди **все способы распределения** целых монет, которые устроят всех.

### Идея

Пусть $\min_x, \min_y, \min_z$ — минимальные доли каждого (округляем вверх). Тогда:
- Если $\min_x + \min_y + \min_z > n$ — пираты не сговорятся, вариантов 0.
- Иначе «свободных» монет $k = n - (\min_x + \min_y + \min_z)$.
- Нужно разложить $k$ на 3 неотрицательных слагаемых — это сочетания с повторениями $\overline{C}(3, k) = C(k+2, 2)$.

### Примеры

**Пример 1.** $n = 100$, $a = 50\%$, $b = 30\%$, $c = 20\%$.
- Минимумы: 50, 30, 20 (сумма = 100).
- Свободных монет: 0.
- Единственный вариант: $(50, 30, 20)$.

**Пример 2.** $n = 100$, $a = 20\%$, $b = 30\%$, $c = 20\%$.
- Минимумы: 20, 30, 20 (сумма = 70).
- Свободных монет: $100 - 70 = 30$.
- Свободные 30 монет можно раскидать по трём пиратам.
- Это задача о разложении 30 на 3 неотрицательных слагаемых — сочетания с повторениями $\overline{C}(3, 30) = C(32, 2) = 496$ вариантов!

In [ ]:
# ============================================================
# Задача 2: Счастливые пираты — перебор распределений
# ============================================================

def pirate_distributions(n, a_pct, b_pct, c_pct):
    """Находим все способы распределения n монет между 3 пиратами.
    a_pct, b_pct, c_pct — проценты, в сумме НЕ больше 100.
    Возвращает список кортежей (x, y, z) — сколько каждому пирату.
    """
    assert a_pct + b_pct + c_pct <= 100, "Сумма процентов не должна превышать 100"

    # Минимальные доли каждого пирата (округляем вверх)
    min_x = -(-n * a_pct // 100)   # ceil(n * a / 100) — трюк с двойным минусом
    min_y = -(-n * b_pct // 100)
    min_z = -(-n * c_pct // 100)

    # Если минимальные требования в сумме больше n — пираты не сговорятся
    if min_x + min_y + min_z > n:
        return []

    distributions = []
    # Перебираем все x от min_x до n (с запасом)
    for x in range(min_x, n + 1):
        # Перебираем y от min_y до n - x
        for y in range(min_y, n - x + 1):
            z = n - x - y           # третьему — остаток
            if z >= min_z:          # проверяем, что третьему хватает
                distributions.append((x, y, z))
    return distributions

# Пример 1: жёсткие проценты — единственное решение
print("--- Пример 1: n=100, a=50%, b=30%, c=20% ---")
results1 = pirate_distributions(100, 50, 30, 20)
print(f"Найдено способов: {len(results1)}")
for r in results1:
    print(f"  {r}")

# Пример 2: есть свобода — много решений
print("\n--- Пример 2: n=100, a=20%, b=30%, c=20% (сумма 70, свободных 30) ---")
results2 = pirate_distributions(100, 20, 30, 20)
print(f"Найдено способов: {len(results2)}")
print("Первые 10:")
for r in results2[:10]:
    print(f"  {r}")
print(f"  ... (всего {len(results2)})")

# Пример 3: маленькая задача — n=10
print("\n--- Пример 3: n=10, a=20%, b=30%, c=40% (сумма 90) ---")
results3 = pirate_distributions(10, 20, 30, 40)
print(f"Найдено способов: {len(results3)}")
for r in results3:
    print(f"  {r}")

# Проверка: для примера 2 — это сочетания с повторениями
# Свободных монет: 100 - (20+30+20) = 30. Разложить 30 на 3 слагаемых: C(30+2, 2) = C(32, 2)
free_coins = 100 - (20 + 30 + 20)
expected = math.comb(free_coins + 3 - 1, 3 - 1)
print(f"\nПроверка по формуле сочетаний с повторениями:")
print(f"  Свободных монет: {free_coins}")
print(f"  C({free_coins}+2, 2) = C({free_coins+2}, 2) = {expected}")
print(f"  Фактически найдено: {len(results2)}")
print(f"  Совпадает: {expected == len(results2)}")

---

## 12. Задача 3 — Bruteforce

### Постановка

Пароль от шкатулки состоит из символов заданного алфавита. Длина пароля **неизвестна, но не больше 20**. Сколько комбинаций надо перебрать в худшем случае?

### Считаем

Пусть алфавит состоит из $n$ символов. Пароль может быть длины 1, 2, ..., 20.

Число паролей длины $m$ — это размещения с повторениями: $n^m$.

**Итого в худшем случае:**

$$\text{Total} = n^1 + n^2 + n^3 + \ldots + n^{20} = \sum_{m=1}^{20} n^m$$

По формуле геометрической прогрессии:

$$\sum_{m=1}^{20} n^m = n \cdot \frac{n^{20} - 1}{n - 1}$$

### Примеры

| Алфавит | $n$ | Худший случай (сумма для m=1..20) |
|---|---|---|
| Цифры | 10 | $\approx 1.11 \times 10^{20}$ |
| Буквы (lowercase) | 26 | $\approx 2 \times 10^{28}$ |
| Буквы + цифры | 36 | $\approx 1.76 \times 10^{31}$ |
| Все печатные символы | 95 | $\approx 6.08 \times 10^{39}$ |

In [ ]:
# ============================================================
# Задача 3: Bruteforce — считаем число комбинаций и подбираем пароль
# ============================================================

def total_combinations(alphabet_size, max_length=20):
    """Считаем суммарное число паролей длины 1..max_length.
    Это сумма n^1 + n^2 + ... + n^max_length.
    По формуле геом. прогрессии: n * (n^max - 1) / (n - 1).
    """
    n = alphabet_size
    if n == 1:
        return max_length   # вырожденный случай
    return n * (n ** max_length - 1) // (n - 1)

# Разные алфавиты
cases = [
    ("Цифры (0-9)",                   10),
    ("Буквы lowercase (a-z)",         26),
    ("Буквы + цифры (a-z, 0-9)",      36),
    ("Буквы обоих регистров + цифры", 62),
    ("Все печатные ASCII",            95),
]

print(f"{'Алфавит':<40} {'n':>4} {'Вариантов (длина 1..20)':>30}")
print("-" * 80)
for name, n in cases:
    total = total_combinations(n, 20)
    print(f"{name:<40} {n:>4} {total:>30,}")

In [ ]:
# ============================================================
# Задача 3 (продолжение): функция подбора пароля bruteforce
# ============================================================
# Демонстрируем на маленьком пароле, иначе ждать придётся вечность

import itertools
import time

def bruteforce_password(target_password, alphabet, max_length=4):
    """Подбираем пароль полным перебором.
    target_password — что ищем (строка).
    alphabet — строка из допустимых символов.
    max_length — максимальная длина для перебора.
    Возвращает (найденный_пароль, число_попыток, время_в_секундах).
    """
    start = time.time()                         # засекаем время
    attempts = 0                                # счётчик попыток

    # Перебираем длину от 1 до max_length
    for length in range(1, max_length + 1):
        # product(alphabet, repeat=length) — размещения с повторениями
        for combo in itertools.product(alphabet, repeat=length):
            attempts += 1                       # ещё одна попытка
            candidate = ''.join(combo)          # собираем строку из кортежа
            if candidate == target_password:
                elapsed = time.time() - start
                return candidate, attempts, elapsed
    return None, attempts, time.time() - start  # не нашли

# Демонстрация 1: 3-значный пин-код
target1 = "742"
alphabet1 = "0123456789"
print(f"--- Подбираем пин-код '{target1}' (алфавит: цифры) ---")
found, attempts, t = bruteforce_password(target1, alphabet1, max_length=4)
print(f"Найден: {found}")
print(f"Попыток: {attempts}")
print(f"Время: {t*1000:.2f} мс\n")

# Демонстрация 2: 4-значный буквенный пароль
target2 = "abcz"
alphabet2 = "abcdefghijklmnopqrstuvwxyz"
print(f"--- Подбираем пароль '{target2}' (алфавит: a-z) ---")
found, attempts, t = bruteforce_password(target2, alphabet2, max_length=4)
print(f"Найден: {found}")
print(f"Попыток: {attempts}")
print(f"Время: {t:.3f} сек\n")

# Демонстрация 3: 5-значный буквенно-цифровой (чтобы показать рост)
target3 = "ab12z"
alphabet3 = "abcdefghijklmnopqrstuvwxyz0123456789"
print(f"--- Подбираем пароль '{target3}' (алфавит: a-z + 0-9) ---")
found, attempts, t = bruteforce_password(target3, alphabet3, max_length=5)
print(f"Найден: {found}")
print(f"Попыток: {attempts}")
print(f"Время: {t:.3f} сек")

# Сколько времени на 8-значный пароль из 70 символов?
n, m = 70, 8
total = n ** m
print(f"\n--- А теперь представь: пароль длины {m} из {n} символов ---")
print(f"Всего комбинаций: {total:,}")
print(f"При 1 млрд попыток/сек: {total / 1e9 / 86400:.1f} дней")
print(f"Профессор: вот почему длинные пароли — это не паранойя, а арифметика.")

In [ ]:
# ============================================================
# Картинка 8: рост числа попыток для разных длин пароля
# ============================================================

lengths = list(range(1, 11))                  # длина пароля от 1 до 10

# Алфавиты: цифры, буквы, буквы+цифры, все символы
alphabets = [('цифры (n=10)', 10, '#e74c3c'),
             ('буквы (n=26)', 26, '#3498db'),
             ('буквы+цифры (n=36)', 36, '#27ae60'),
             ('все символы (n=70)', 70, '#9b59b6')]

fig, ax = plt.subplots(figsize=(11, 6))
ax.set_yscale('log')                            # логарифмическая шкала

for name, n, color in alphabets:
    vals = [n ** m for m in lengths]
    ax.plot(lengths, vals, 'o-', color=color, linewidth=2.2, markersize=8, label=name)

# Подсветим "взламываемые за секунду" — до 1e9
ax.axhspan(1, 1e9, alpha=0.1, color='red', label='взлом за <1 сек (1 млрд/сек)')

ax.set_xlabel('длина пароля', fontsize=12)
ax.set_ylabel('число комбинаций', fontsize=12)
ax.set_title('Bruteforce: сколько комбинаций у пароля разной длины\nЛогарифмическая шкала',
             fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3, which='both')
ax.set_xticks(lengths)
plt.tight_layout()
plt.show()

---

## 13. Зачем вся эта комбинаторика в ML

Кажется, что комбинаторика — это про пароли и лотереи. Но на самом деле **вся современная ML-практика** пронизана подсчётами.

### 1. Grid Search по гиперпараметрам

У тебя 5 гиперпараметров с диапазонами:

| Параметр | Значений |
|---|---|
| learning rate | 6 |
| batch size | 4 |
| число слоёв | 5 |
| dropout | 3 |
| optimizer | 3 |

**Итого:** $6 \cdot 4 \cdot 5 \cdot 3 \cdot 3 = 1080$ конфигураций. Если одна обучение занимает 30 минут — это **22.5 дня**. Без комбинаторики ты бы не оценил это заранее.

### 2. Random Search и сэмплинг

Если конфигураций слишком много — берём случайную выборку из них. Сколько комбинаций пропустим? — опять комбинаторика.

### 3. Ансамбли моделей

Выбрать топ-3 модели из 10 для голосования:
- Если порядок не важен (мажоритарное голосование): $C(10, 3) = 120$.
- Если важен (1-я — основной, 2-я — fallback, 3-я — корректировка): $A(10, 3) = 720$.

### 4. Аугментации данных

Картинку можно:
- повернуть (8 углов),
- сдвинуть (4 направления),
- изменить цвет (5 фильтров),
- добавить шум (3 уровня).

**Итого:** $8 \cdot 4 \cdot 5 \cdot 3 = 480$ аугментаций одной картинки. Это комбинаторный взрыв.

### 5. Cross-validation

Разбить $n$ точек на $k$ folds: число способов = $\frac{n!}{(n/k)!^k \cdot k!}$. При $n = 100, k = 5$ — это $\approx 10^{67}$ разбиений. Поэтому на практике берут несколько случайных.

### 6. Топологии нейросетей

Сколько разных архитектур можно построить из $L$ слоёв, каждый из $k$ типов? Это размещения с повторениями: $k^L$. При $L = 10, k = 5$ — $5^{10} = 9\,765\,625$ архитектур. Это почему NAS (Neural Architecture Search) — сложная задача.

### 7. Байесовские модели

В Байесовском выводе часто нужно считать $C(n, m)$ для выбора подмножества признаков. Это и есть биномиальный коэффициент.

> **Главный вывод:** если ты не понимаешь, **сколько вариантов** у твоей модели/алгоритма/датасета — ты не управляешь процессом. Ты merely "экспериментируешь", пока не кончатся деньги на AWS.

---

## 14. Сводная шпаргалка

| Тип | Порядок | Повторения | Формула | Пример |
|---|---|---|---|---|
| Перестановка | да | нет | $n!$ | 5 гостей по 5 стульям = 120 |
| Размещение | да | нет | $\frac{n!}{(n-m)!}$ | 3 должности из 10 человек = 720 |
| Размещение с повт. | да | да | $n^m$ | 4-значный пин = 10 000 |
| Сочетание | нет | нет | $\frac{n!}{m!(n-m)!}$ | 3 пиццы из 10 = 120 |
| Сочетание с повт. | нет | да | $\frac{(n+m-1)!}{m!(n-1)!}$ | 3 лимонада из 5 (с повторами) = 35 |

### Три вопроса, чтобы выбрать формулу

1. **Берём все элементы или только часть?**
   - Все → перестановка.
   - Часть → размещение или сочетание.
2. **Порядок важен?**
   - Да → размещение (или перестановка).
   - Нет → сочетание.
3. **Элементы могут повторяться?**
   - Да → берём версию «с повторениями».
   - Нет → обычная версия.

### Полезные тождества

- $C(n, m) = C(n, n-m)$ — симметрия.
- $C(n, 0) = C(n, n) = 1$.
- $\sum_{m=0}^{n} C(n, m) = 2^n$ — всего подмножеств множества из $n$ элементов (включая пустое).
- $C(n, m) = C(n-1, m-1) + C(n-1, m)$ — треугольник Паскаля.
- $0! = 1$ — особый случай.

---

## 15. Итоговый тест — 10 вопросов

> Запусти ячейку ниже (Shift+Enter). Все вопросы — простым языком, **без формул в вариантах ответа**. Везде используются Unicode-символы (∈, !, С, A), чтобы рендерилось в любом браузере и Colab.

In [ ]:
# ============================================================
# Интерактивный тест — все вопросы и варианты плейнтекстом
# ============================================================
# Запусти ячейку (Shift+Enter), чтобы начать тест.

import ipywidgets as widgets
from IPython.display import display, clear_output

# Вопросы: (вопрос, [варианты], индекс правильного, пояснение)
# ВАЖНО: всё плейнтекстом с Unicode — НЕТ LaTeX, чтобы рендерилось везде
QUESTIONS = [
    {
        'q': 'Чему равно 5! (пять факториал)?',
        'options': ['25', '100', '120', '720'],
        'correct': 2,
        'expl': '5! = 5*4*3*2*1 = 120. Это, например, число способов рассадить 5 гостей по 5 стульям.'
    },
    {
        'q': 'Чему равно 0! (ноль факториал)?',
        'options': ['0', '1', 'не определено', 'бесконечность'],
        'correct': 1,
        'expl': '0! = 1 — особый случай. Нужен, чтобы формулы вроде C(n, n) = 1 работали без исключений.'
    },
    {
        'q': 'Сколько существует перестановок множества из 4 элементов?',
        'options': ['4', '16', '24', '256'],
        'correct': 2,
        'expl': 'Перестановки = n! = 4! = 4*3*2*1 = 24.'
    },
    {
        'q': 'Ты выбираешь 3 пиццы из 10 (порядок не важен). Это какой тип задачи?',
        'options': [
            'Перестановка — P(10, 3)',
            'Сочетание — C(10, 3)',
            'Размещение — A(10, 3)',
            'Размещение с повторениями — 10^3',
        ],
        'correct': 1,
        'expl': 'Порядок не важен (в корзине всё равно, в каком порядке лежат пиццы) → это сочетание. C(10, 3) = 120.'
    },
    {
        'q': 'Чему равно C(10, 3) (число сочетаний 3 из 10)?',
        'options': ['30', '120', '720', '1000'],
        'correct': 1,
        'expl': 'C(10, 3) = 10! / (3! * 7!) = (10*9*8) / (3*2*1) = 720 / 6 = 120.'
    },
    {
        'q': 'Ты выбираешь президента, вице- и секретаря из 10 человек. Это:',
        'options': [
            'Сочетание — порядок не важен',
            'Размещение — порядок важен, все разные',
            'Перестановка — берём всех',
            'Сочетание с повторениями',
        ],
        'correct': 1,
        'expl': 'Должности разные → порядок важен. Не всех берём (3 из 10) → это размещение A(10, 3) = 720.'
    },
    {
        'q': 'Сколько всего 4-значных пин-кодов (цифры 0-9 могут повторяться)?',
        'options': ['1 000', '10 000', '5 040', '100 000'],
        'correct': 1,
        'expl': 'Размещение с повторениями: n^m = 10^4 = 10 000. Каждая позиция независима, цифры могут повторяться.'
    },
    {
        'q': 'Сколько счастливых билетов из 6 цифр (сумма первых 3 = сумме последних 3)?',
        'options': ['5 252', '55 252', '502 525', '5 525 252'],
        'correct': 1,
        'expl': 'Ответ — 55 252. Мы посчитали это в задаче 1 перебором всех 1 000 000 билетов.'
    },
    {
        'q': 'Что больше: A(10, 3) или C(10, 3), и во сколько раз?',
        'options': [
            'A(10, 3) больше в 2 раза',
            'A(10, 3) больше в 3! = 6 раз',
            'C(10, 3) больше в 6 раз',
            'Они равны',
        ],
        'correct': 1,
        'expl': 'A(10, 3) = 720, C(10, 3) = 120. Размещений больше в 6 раз = 3!. Для каждой тройки есть 3! = 6 порядков.'
    },
    {
        'q': 'В чём разница между ПОДСЧЁТОМ и ПЕРЕЧИСЛЕНИЕМ?',
        'options': [
            'Подсчёт = "сколько вариантов", перечисление = "какие именно"',
            'Подсчёт = для маленьких задач, перечисление = для больших',
            'Это одно и то же',
            'Подсчёт точнее, перечисление приближённое',
        ],
        'correct': 0,
        'expl': 'Подсчёт отвечает на "СКОЛЬКО" (число), перечисление — на "КАКИЕ" (список). Например, 52! посчитать легко, а выписать невозможно.'
    },
]

# Состояние теста
state = {'index': 0, 'score': 0, 'answers': []}

# Виджеты
question_html = widgets.HTML()
options_radio = widgets.RadioButtons(options=[], layout=widgets.Layout(width='80%'))
feedback_html = widgets.HTML()
next_button = widgets.Button(description='Ответить →', button_style='primary',
                              layout=widgets.Layout(width='200px'))
progress_label = widgets.HTML()

def render_question():
    """Отрисовка текущего вопроса."""
    i = state['index']
    q = QUESTIONS[i]
    question_html.value = (
        f'<h3 style="color:#2c3e50">Вопрос {i+1} из {len(QUESTIONS)}</h3>'
        f'<p style="font-size:14px">{q["q"]}</p>'
    )
    options_radio.options = q['options']
    options_radio.value = None
    feedback_html.value = ''
    next_button.description = 'Ответить →'
    progress_label.value = f'<small>Счёт: {state["score"]}/{state["index"]}</small>'

def on_next(b):
    """Обработка нажатия на кнопку Ответить."""
    i = state['index']
    q = QUESTIONS[i]

    if options_radio.value is None:
        feedback_html.value = '<span style="color:#e74c3c">⚠️ Выбери вариант ответа!</span>'
        return

    chosen = q['options'].index(options_radio.value)
    correct = chosen == q['correct']

    if correct:
        state['score'] += 1
        feedback_html.value = (
            f'<div style="background:#eafaf1;padding:10px;border-left:4px solid #27ae60">'
            f'<b style="color:#27ae60">✓ Верно!</b><br>'
            f'<small>{q["expl"]}</small></div>'
        )
    else:
        correct_opt = q['options'][q['correct']]
        feedback_html.value = (
            f'<div style="background:#fdedee;padding:10px;border-left:4px solid #e74c3c">'
            f'<b style="color:#e74c3c">✗ Неверно.</b> Правильный ответ: <b>{correct_opt}</b><br>'
            f'<small>{q["expl"]}</small></div>'
        )

    state['answers'].append({
        'q': q['q'], 'chosen': chosen,
        'correct': q['correct'], 'ok': correct
    })

    if i + 1 < len(QUESTIONS):
        next_button.description = 'Следующий вопрос →'
        next_button.on_click(go_next, remove=True)
        next_button.on_click(go_next)
    else:
        next_button.description = 'Показать результат 🏁'
        next_button.on_click(show_results, remove=True)
        next_button.on_click(show_results)

def go_next(b):
    """Переход к следующему вопросу."""
    state['index'] += 1
    next_button.on_click(go_next, remove=True)
    next_button.on_click(on_next)
    render_question()

def show_results(b):
    """Финальный экран с результатом."""
    score = state['score']
    total = len(QUESTIONS)
    pct = score / total * 100

    if pct >= 90:
        grade, color, emoji = 'Отлично!', '#27ae60', '🎉'
    elif pct >= 70:
        grade, color, emoji = 'Хорошо', '#3498db', '👍'
    elif pct >= 50:
        grade, color, emoji = 'Удовлетворительно', '#f39c12', '🤔'
    else:
        grade, color, emoji = 'Нужно перечитать', '#e74c3c', '📚'

    html = f'''
    <div style="background:{color};color:white;padding:20px;border-radius:10px;text-align:center">
        <h2>{emoji} {grade}</h2>
        <p style="font-size:18px">Твой результат: <b>{score} / {total}</b> ({pct:.0f}%)</p>
    </div>
    <h4 style="margin-top:20px">Разбор ответов:</h4>
    <ol>
    '''
    for ans in state['answers']:
        icon = '✓' if ans['ok'] else '✗'
        clr = '#27ae60' if ans['ok'] else '#e74c3c'
        html += f'<li style="color:{clr}">{icon} {ans["q"][:80]}...</li>'
    html += '</ol>'

    clear_output()
    display(widgets.HTML(html))

# Стартуем тест
next_button.on_click(on_next)
render_question()

display(widgets.VBox([
    progress_label,
    question_html,
    options_radio,
    feedback_html,
    next_button,
]))

---

## Финал лекции

> Вот и всё, студент! Ты прошёл:
>
> - факториал и его свойства (включая $0! = 1$)
> - три типа комбинаций: перестановки, размещения, сочетания
> - версии с повторениями (для паролей и покупок «с повторами»)
> - разницу между подсчётом и перечислением
> - три задачи из лекции ВК: счастливые билеты (55 252), счастливые пираты, bruteforce
> - связь всей этой математики с ML: grid search, ансамбли, аугментации
> - тест на 10 вопросов
>
> Если прошёл тест на 7+ — поздравляю, ты освоил фундамент комбинаторики. Дальше пойдёт **теория вероятностей** — и там все эти формулы оживут: мы будем делить «число удачных исходов» на «общее число исходов», а это ровно наши $C$ и $A$.
>
> Если не прошёл — ничего страшного. Перечитай раздел с примерами, посмотри на картинки. Комбинаторика — это не про формулы, а про **внимательно смотреть на задачу** и задавать три вопроса: «все или часть?», «порядок важен?», «повторения есть?».
>
> Удачи! *Профессор.*

---

## Полезные ссылки

- **Курс ВК «Математика для ML»:** https://education.vk.company/curriculum/program/lesson/31731/
- **AllCups (задания курса):** https://allcups.run/
- **itertools — документация:** https://docs.python.org/3/library/itertools.html
- **Треугольник Паскаля (Википедия):** https://ru.wikipedia.org/wiki/Треугольник_Паскаля

> *Ноутбук создан по материалам Лекции 2 курса «Математика для машинного обучения и анализа данных» (ВКонтакте Образование, 2025).*
> *Архитектура повторяет v3 Лекции 1 — жизненные примеры, простые формулы, Unicode-символы в тесте, matplotlib-визуализации, комментарии на каждой строке кода.*